# AeroIntel — 02 Train YOLO (Colab, break-safe)

Trains **YOLO11s** on the official dataset — **AeroIntel-ABC-v1** = `datasets/master_dataset_ABC`
(4 frozen classes: `0 Crack, 1 Corrosion, 2 Dent, 3 Missing Fastener`) — spec **A6 step 5**.

**Why YOLO11s:** Ultralytics' current generation; better accuracy per GPU-hour than YOLOv8 at the
same size, comfortably fits a free Colab T4 at imgsz=640 / batch=16, and exports cleanly to ONNX
for the R2 detector service. (If you want a lighter/faster baseline, set `MODEL_SIZE = 'n'`.)

## How the "training with breaks" works
Colab free runtimes disconnect. This notebook is built to survive that:

1. **Chunked epochs** — training runs in chunks (default `EPOCHS_PER_CHUNK = 10`). After each chunk the checkpoint `last.pt` is **copied to Google Drive** (cell 6).
2. **Auto-resume** — if the runtime died mid-chunk, just **Runtime → Run all**. It restores the dataset and the run folder from Drive and resumes via `resume=True` (optimizer state + epoch count from `last.pt`).
3. **Nothing lives only in /content** — dataset, checkpoints, logs and best weights are all in Drive.

> Do NOT change `TOTAL_EPOCHS` between chunks, and do not edit anything in the run folder — resume reads `args.yaml` + `last.pt` from there.

In [ ]:
# @title 1. Mount Drive + install ultralytics
from google.colab import drive
drive.mount('/content/drive')

%pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# @title 2. Config — all knobs in one place
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AeroIntel')

# Upload aerointel_dataset_v1_colab.zip to MyDrive/AeroIntel/datasets/
# (the zip was rebuilt from datasets/master_dataset_ABC with Linux-safe paths).
# The old name is still accepted in case only that one is uploaded.
ZIP_CANDIDATES = [
    DRIVE_ROOT / 'datasets/aerointel_dataset_v1_colab.zip',
    DRIVE_ROOT / 'datasets/aerointel_dataset_v1.zip',
]

DATA_ROOT = '/content/aerointel'          # fast local disk
DATA_YAML = f'{DATA_ROOT}/data.yaml'      # zip has data.yaml at its root

MODEL_SIZE   = 's'     # @param ['s', 'n']  -> YOLO11s first; 'n' for a faster/lighter run
TOTAL_EPOCHS = 100     # @param {type:"integer"}  # A6: 100 epochs + early stopping
EPOCHS_PER_CHUNK = 10  # @param {type:"integer"}  # size of each break-safe chunk
IMGSZ        = 640     # @param {type:"integer"}  # A6: imgsz=640
BATCH        = 16      # @param {type:"integer"}  # lower to 8 if you hit OOM on T4
PATIENCE     = 20      # @param {type:"integer"}  # early stopping (A6)
WORKERS      = 2       # Colab-safe dataloader workers

RUN_NAME  = f'aerointel_v1_yolo11{MODEL_SIZE}'
RUNS_DIR  = DRIVE_ROOT / 'runs'           # Drive copy (permanent)
WORK_RUNS = Path('/content/runs')         # local working copy (fast)

assert EPOCHS_PER_CHUNK <= TOTAL_EPOCHS
print(f'Model: yolo11{MODEL_SIZE} | epochs: {TOTAL_EPOCHS} in chunks of {EPOCHS_PER_CHUNK} | imgsz={IMGSZ} batch={BATCH}')
print('Run name:', RUN_NAME)

In [ ]:
# @title 3. Unzip dataset from Drive into fast local disk (+ verify)
import zipfile

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
assert zip_path, ('No dataset zip found in Drive. Upload ONE of:\n  '
                  + '\n  '.join(str(p) for p in ZIP_CANDIDATES))
print('Using zip:', zip_path)

if not Path(DATA_YAML).exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_ROOT)

# auto-detect the zip layout: data.yaml at the root (master_dataset_ABC layout)
# or nested under merged/ (older notebook-01 layout). Then verify the splits.
root = Path(DATA_ROOT)
if (root / 'data.yaml').exists():
    DATA_YAML = str(root / 'data.yaml')
elif (root / 'merged/data.yaml').exists():
    DATA_YAML = str(root / 'merged/data.yaml')
    root = root / 'merged'
else:
    raise AssertionError('data.yaml not found after unzip — bad zip layout')

for split in ('train', 'valid', 'test'):
    n_img = len(list((root / split / 'images').glob('*')))
    assert n_img > 0, f'{split}/images is empty — zip extracted wrong'
    print(f'{split}: {n_img} images')

print('\nDataset ready:', DATA_YAML)
!sed -n 1,10p "{DATA_YAML}\"

In [ ]:
# @title 4. Load resume state from Drive (where did training stop?)
import json

STATE_FILE = RUNS_DIR / f'{RUN_NAME}_state.json'
DRIVE_LAST  = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
DRIVE_BEST  = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'

def read_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {'epochs_done': 0, 'chunk': 0}

state = read_state()
print(f'Run: {RUN_NAME}')
print(f"Epochs completed: {state['epochs_done']} / {TOTAL_EPOCHS}")
print(f"Checkpoint on Drive: {'YES' if DRIVE_LAST.exists() else 'no (fresh start)'}")
if state['epochs_done'] >= TOTAL_EPOCHS:
    print('\nThis run is already finished. To train the comparison model, change MODEL_SIZE above (s -> n) and rerun from cell 2.')

In [ ]:
# @title 4b. (Optional, 1–2 min) Speed probe — predict YOUR real training time
# Runs a throwaway mini-train (1 epoch on 5% of the data) to measure GPU speed.
# Does NOT touch your real run or resume state.
import time
from pathlib import Path
from ultralytics import YOLO

n_train = len(list((Path(DATA_ROOT) / 'train/images').glob('*')))
print(f'Train images: {n_train}')
probe = YOLO(f'yolo11{MODEL_SIZE}.pt')
# warmup run (downloads weights + scans dataset — excluded from timing)
probe.train(data=DATA_YAML, epochs=1, fraction=0.05, imgsz=IMGSZ, batch=BATCH,
            project=str(WORK_RUNS), name='throughput_probe', exist_ok=True,
            plots=False, val=False, verbose=False)
# timed run with warm caches -> realistic per-epoch speed
t0 = time.time()
probe.train(data=DATA_YAML, epochs=1, fraction=0.05, imgsz=IMGSZ, batch=BATCH,
            project=str(WORK_RUNS), name='throughput_probe', exist_ok=True,
            plots=False, val=False, verbose=False)
dt = time.time() - t0
sec_per_epoch = dt / 0.05
print(f'\nProbe: {dt:.0f}s for 5% of one epoch -> ~{sec_per_epoch/60:.1f} min/epoch')
print(f'Estimated TOTAL: ~{sec_per_epoch * TOTAL_EPOCHS / 3600:.1f} h for {TOTAL_EPOCHS} epochs (add ~10% for per-epoch validation; early stopping usually ends it sooner)')
print(f'Per chunk of {EPOCHS_PER_CHUNK} epochs: ~{sec_per_epoch * EPOCHS_PER_CHUNK / 60:.0f} min')

In [ ]:
# @title 5. TRAIN one chunk (re-run this cell until it prints ALL EPOCHS DONE)
import json, shutil
from ultralytics import YOLO

local_run  = WORK_RUNS / RUN_NAME
local_last = local_run / 'weights' / 'last.pt'

# restore the local working copy from Drive (fresh runtime / re-run)
if DRIVE_LAST.exists() and not local_last.exists():
    local_run.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUNS_DIR / RUN_NAME, local_run, dirs_exist_ok=True)
    print('Restored run folder from Drive (resume ready).')

state_before = read_state()   # read BEFORE training so a crashed chunk can't lose the counter

def train_chunk():
    if state_before['epochs_done'] >= TOTAL_EPOCHS:
        print('ALL EPOCHS DONE for this run — nothing to train.')
        return
    resume = DRIVE_LAST.exists() and state_before['epochs_done'] > 0
    model = YOLO(str(DRIVE_LAST)) if resume else YOLO(f'yolo11{MODEL_SIZE}.pt')
    print(f"\n=== Chunk {state_before['chunk'] + 1}: epochs {state_before['epochs_done'] + 1}–"
          f"{min(state_before['epochs_done'] + EPOCHS_PER_CHUNK, TOTAL_EPOCHS)} "
          f"({'RESUME' if resume else 'FRESH START'}) ===")
    try:
        model.train(
            data=DATA_YAML,
            epochs=TOTAL_EPOCHS,
            resume=resume,
            imgsz=IMGSZ,
            batch=BATCH,
            patience=PATIENCE,
            workers=WORKERS,
            project=str(WORK_RUNS),
            name=RUN_NAME,
            exist_ok=True,
            seed=42,
            deterministic=True,
            verbose=True,
        )
    except Exception as e:
        # ultralytics strips the optimizer from last.pt once a run finishes or
        # early-stops -> resume can fail. Fall back to a weights-only warm start.
        if resume and ('optimizer' in str(e).lower() or 'resume' in str(e).lower()):
            warm = DRIVE_BEST if DRIVE_BEST.exists() else local_run / 'weights' / 'best.pt'
            assert warm.exists(), f'Resume failed and no best.pt found: {e}'
            print(f'\nResume failed ({e})')
            print(f'-> FALLBACK: weights-only warm start from {warm} (epoch counter restarts).')
            YOLO(str(warm)).train(
                data=DATA_YAML, epochs=TOTAL_EPOCHS, imgsz=IMGSZ, batch=BATCH,
                patience=PATIENCE, workers=WORKERS, project=str(WORK_RUNS),
                name=RUN_NAME, exist_ok=True, seed=42, verbose=True,
            )
        else:
            raise

train_chunk()

# report where we are now (the trainer writes its own args/progress into the run folder)
results_csv = local_run / 'results.csv'
epochs_now = 0
if results_csv.exists():
    epochs_now = max(0, len(results_csv.read_text().strip().splitlines()) - 1)
new_state = {'epochs_done': epochs_now, 'chunk': state_before['chunk'] + 1}
STATE_FILE.write_text(json.dumps(new_state, indent=2))
print(f"\nEpochs completed so far: {epochs_now}/{TOTAL_EPOCHS}")
if epochs_now >= TOTAL_EPOCHS:
    print('ALL EPOCHS DONE — go to cell 6 (sync) then notebook 03.')
else:
    print(f'Run this cell again to train the next chunk of {EPOCHS_PER_CHUNK} epochs.')

In [ ]:
# @title 6. Sync checkpoints to Drive (run after EVERY chunk / before you close the tab)
import shutil
from pathlib import Path

local_run = WORK_RUNS / RUN_NAME
assert (local_run / 'weights' / 'last.pt').exists(), 'No local checkpoint found — train first'

(RUNS_DIR / RUN_NAME).mkdir(parents=True, exist_ok=True)
# copy everything except bulky cache files
for item in local_run.rglob('*'):
    if item.suffix in {'.cache'} or item.name in {'.DS_Store'}:
        continue
    rel = item.relative_to(local_run)
    dst = RUNS_DIR / RUN_NAME / rel
    if item.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dst)
print('Synced to Drive:', RUNS_DIR / RUN_NAME)
print('Safe to close the tab now — training will resume from here later.')

## The break loop
Repeat until cell 5 prints **ALL EPOCHS DONE**:

1. Run cell 5 (one chunk ≈ a few minutes on a T4 depending on dataset size).
2. Run cell 6 (sync to Drive). *Cell 5 alone is not durable — Drive is.*
3. Take your break. If the runtime dies: reopen, **Runtime → Run all** — cells 1–4 restore everything, cell 5 resumes the next chunk automatically.

### If the GPU session dies mid-chunk
Nothing is lost except the current chunk. Run all again — `resume=True` restores the optimizer state and epoch count from `last.pt`.

### If `resume` errors after an ultralytics version change (or early stopping)
Cell 5 catches it and falls back to a **weights-only warm start from `best.pt`** (epoch counter restarts; shrink `TOTAL_EPOCHS` if you only want a few more epochs). Note it in `docs/DECISIONS.md`.

In [ ]:
# @title 7. (Optional) Training curves
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(WORK_RUNS / RUN_NAME / 'results.csv')
df.columns = [c.strip() for c in df.columns]
metrics = ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for a, m in zip(ax, metrics):
    a.plot(df['epoch'], df[m]); a.set_title(m.split('/')[-1]); a.set_xlabel('epoch')
plt.tight_layout(); plt.show()